In [2]:
import duckdb as dd

In [3]:
# Create a persistent connection to duckdb to save data for use later. Extracted folder should be labelled the same, otherwise file path can be changed to what contains the json objects.
conn = dd.connect('spotify_all_years.db')

In [ ]:
#First table: combine all year's JSON files into one
conn.execute(
"CREATE TABLE complete_data AS " \
"SELECT * " \
"FROM read_json_auto('Spotify Extended Streaming History/*.json', union_by_name=true)"
)

In [ ]:
#Inspect data
conn.execute("SELECT * FROM complete_data LIMIT 5").df()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2016-12-07 22:36:35,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",144244,US,97.71.20.155,Juju on That Beat (TZ Anthem),Zay Hilfigerrr,Juju on That Beat (TZ Anthem),spotify:track:1lItf5ZXJc1by9SbPeljFd,None,...,None,None,None,trackdone,trackdone,True,False,False,<NA>,False
1,2016-12-07 22:37:28,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",50521,US,97.71.20.155,"i hate u, i love u (feat. olivia o'brien)",gnash,us,spotify:track:7vRriwrloYVaoAe3a9wJHe,None,...,None,None,None,trackdone,fwdbtn,True,False,False,<NA>,False
2,2016-12-07 22:41:17,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",230453,US,97.71.20.155,Starboy,The Weeknd,Starboy,spotify:track:7MXVkk9YMctZqd1Srtv4MB,None,...,None,None,None,fwdbtn,trackdone,True,False,False,<NA>,False
3,2016-12-07 22:54:06,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",214480,US,97.71.20.155,Don't Wanna Know,Maroon 5,Don't Wanna Know,spotify:track:0AKejOKlGdiB53QpwAeenO,None,...,None,None,None,trackdone,trackdone,True,False,False,<NA>,False
4,2016-12-07 22:54:13,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",5022,US,97.71.20.155,One Call Away,Charlie Puth,Nine Track Mind,spotify:track:7soJgKhQTO8hLP2JPRkL5O,None,...,None,None,None,trackdone,fwdbtn,True,False,False,<NA>,False


In [ ]:
# Check how many columns + non-null values 
conn.execute("SELECT * FROM complete_data").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 101657 entries, 0 to 101656
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 101657 non-null  datetime64[us]
 1   platform                           101657 non-null  str           
 2   ms_played                          101657 non-null  int64         
 3   conn_country                       101657 non-null  str           
 4   ip_addr                            101657 non-null  str           
 5   master_metadata_track_name         100994 non-null  str           
 6   master_metadata_album_artist_name  100994 non-null  str           
 7   master_metadata_album_album_name   100994 non-null  str           
 8   spotify_track_uri                  100994 non-null  str           
 9   episode_name                       597 non-null     str           
 10  episode_show_name              

In [ ]:
#remove columns we won't use and make a new table
conn.execute("CREATE TABLE clean_data AS SELECT ts,platform, ms_played, ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,reason_end,shuffle,skipped FROM complete_data")

In [ ]:
#verify columns were dropped
conn.execute("SELECT * FROM clean_data").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 101657 entries, 0 to 101656
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 101657 non-null  datetime64[us]
 1   platform                           101657 non-null  str           
 2   ms_played                          101657 non-null  int64         
 3   ip_addr                            101657 non-null  str           
 4   master_metadata_track_name         100994 non-null  str           
 5   master_metadata_album_artist_name  100994 non-null  str           
 6   master_metadata_album_album_name   100994 non-null  str           
 7   spotify_track_uri                  100994 non-null  str           
 8   reason_start                       101657 non-null  str           
 9   reason_end                         101657 non-null  str           
 10  shuffle                        

In [11]:
# dicrepancy between # entries and tracks because podcasts are still counted.
# remove podcasts/episodes from entries, Ran against just one column for optimization.
conn.execute("CREATE TABLE podcast_dropped AS SELECT * FROM clean_data WHERE master_metadata_album_artist_name IS NOT NULL")
#conn.execute("CREATE TABLE podcast_dropped AS SELECT * FROM clean_data WHERE master_metadata_track_name IS NOT NULL AND master_metadata_album_artist_name IS NOT NULL AND master_metadata_album_album_name IS NOT NULL AND spotify_track_uri IS NOT NULL")
#Another way to perform the same query but would alter original table:
#conn.execute("DELETE FROM clean_data WHERE master_metadata_track_name IS NULL AND master_metadata_album_artist_name IS NULL AND master_metadata_album_album_name IS NULL AND spotify_track_uri IS NULL ")

In [3]:
#verify # of entries went down (meaning podcasts and episodes removed)
conn.execute("SELECT * FROM podcast_dropped").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 100994 entries, 0 to 100993
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 100994 non-null  datetime64[us]
 1   platform                           100994 non-null  str           
 2   ms_played                          100994 non-null  int64         
 3   ip_addr                            100994 non-null  str           
 4   master_metadata_track_name         100994 non-null  str           
 5   master_metadata_album_artist_name  100994 non-null  str           
 6   master_metadata_album_album_name   100994 non-null  str           
 7   spotify_track_uri                  100994 non-null  str           
 8   reason_start                       100994 non-null  str           
 9   reason_end                         100994 non-null  str           
 10  shuffle                        

### Table 1 : Unique songs + unique albums per artist

In [6]:
# num of diff songs and diff albums listened to by artist.
# NOT accurate includes released singles as albums.
conn.execute("CREATE TABLE unique_songs_and_artists_per_artist AS SELECT master_metadata_album_artist_name, COUNT(DISTINCT master_metadata_album_album_name) AS unique_albums_for_artist, COUNT(DISTINCT master_metadata_track_name) AS num_unique_songs_for_artist FROM podcast_dropped GROUP BY master_metadata_album_artist_name")

CatalogException: Catalog Error: Table with name "unique_songs_and_artists_per_artist" already exists!

In [ ]:
# distinct tracks per album per artist
# Limitation: if a user listened to an artist but only 1 song of an album it will not be included because it is considering it a single
conn.execute("SELECT master_metadata_album_artist_name AS artist_name, master_metadata_album_album_name AS album_name, COUNT(DISTINCT master_metadata_track_name) AS num_tracks " \
"FROM podcast_dropped " \
"GROUP BY artist_name, album_name " \
"HAVING num_tracks > 1").df()

,artist_name,album_name,num_tracks
0,Joji,In Tongues (Deluxe),5
1,blackbear,deadroses,2
2,Ellie Goulding,Delirium,3
3,Katy Perry,PRISM,5
4,Juice WRLD,Death Race For Love - Bonus Track Version,8
...,...,...,...
973,NLE Choppa,SLUT SZN,5
974,Cavetown,Animal Kingdom,2
975,Kanye West,Donda,3
976,Zach Bryan,Zach Bryan,2


### Table 2: Count of non-single albums per artists

In [8]:
#One row per artist, with a count of only the albums where you listened to more than 1 track. Non-single albumbs per artist.
conn.execute("CREATE TABLE non_single_albums_per_artist AS" \
" SELECT master_metadata_album_artist_name, COUNT(album_name) AS num_true_albums" \
" FROM (SELECT master_metadata_album_artist_name, master_metadata_album_album_name AS album_name, COUNT(DISTINCT master_metadata_track_name) AS num_tracks FROM podcast_dropped GROUP BY master_metadata_album_artist_name, album_name HAVING num_tracks > 1) AS album_counts" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,487


### Table 3: Play count by artist

In [9]:
# Total play count per artist
conn.execute("CREATE TABLE play_count AS SELECT COUNT(*) AS play_count_per_artist,master_metadata_album_artist_name"\
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,2733


### Table 4: Listening Consistency

In [10]:
# Listening consistency — COUNT(DISTINCT month/year) per artist. Strongest superfan signal you have.
# artist_name | months_active (Jan2019, Feb2019, March2020 -> counted up)
# wallows     | 18
conn.execute("CREATE TABLE listening_consistency AS SELECT master_metadata_album_artist_name, COUNT(DISTINCT strftime('%Y-%m', ts)) AS listening_consistency " \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,2733


### Table 5: Skip Rate

In [11]:
# Skip rate per artist, higher = you skip the artist most of the time, lower = you let them play out.
conn.execute("CREATE TABLE skip_rate AS SELECT master_metadata_album_artist_name, AVG(CAST(skipped AS INT)) as skip_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()


,Count
0,2733


### Table 6: Intentional Listening Rate

In [12]:
# Intentional listening rate: percentage of plays where reason_start IN ('clickrow', 'playbtn', 'backbtn') per artist.
conn.execute("CREATE TABLE intentional_listening_rate AS SELECT master_metadata_album_artist_name, SUM(CAST(reason_start IN ('clickrow', 'playbtn', 'backbtn') AS INT)) / COUNT(*) AS intentional_listening_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,2733


### Table 7 : Completion Rate

In [13]:
# Completion rate — AVG(ms_played) per artist as a proxy (you don't have track duration, so average ms played is the best you can do).
conn.execute("CREATE TABLE completion_rate AS SELECT master_metadata_album_artist_name, SUM(CAST(reason_end = 'trackdone' AS INT)) / COUNT(*) AS completion_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df() 

,Count
0,2733


### Table 8: Revisit-Rate

In [ ]:
conn.execute(" SELECT master_metadata_album_artist_name, ts, datediff('day', previous_ts, ts) AS days_diff" \
" FROM (SELECT master_metadata_album_artist_name, strftime('%Y-%m-%d', ts) AS date,LAG (ts, 1) OVER(PARTITION BY master_metadata_album_artist_name ORDER BY ts) AS previous_ts) AS play_gaps FROM podcast_dropped").df()

In [36]:
conn.execute("SELECT master_metadata_album_artist_name, strftime('%Y-%m-%d', ts) AS date," \
" LAG (ts, 1) OVER(PARTITION BY master_metadata_album_artist_name ORDER BY ts) AS previous_ts, datediff('day', previous_ts, ts) AS days_diff" \
" FROM podcast_dropped").df()


,master_metadata_album_artist_name,date,previous_ts,days_diff
0,24kGoldn,2020-02-10,NaT,<NA>
1,24kGoldn,2020-02-10,2020-02-10 12:04:34,0
2,24kGoldn,2020-02-11,2020-02-10 19:25:23,1
3,24kGoldn,2020-02-11,2020-02-11 01:29:23,0
4,24kGoldn,2020-02-11,2020-02-11 04:32:50,0
...,...,...,...,...
100989,softy,2024-07-22,NaT,<NA>
100990,xaretti,2022-10-31,NaT,<NA>
100991,揽佬SKAI ISYOURGOD,2024-12-02,NaT,<NA>
100992,揽佬SKAI ISYOURGOD,2024-12-02,2024-12-02 23:50:45,0


In [ ]:
conn.execute("CREATE TABLE revisit_rate AS SELECT sum AS revisit_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name")

In [ ]:
conn.execute("SELECT * FROM revisit_rate").df().info()

In [ ]:
# Discography coverage over time: did you discover and work through this artist's catalog progressively over time or did you hear it all albums in one week binge
''' We will track time elapsed between first album heard and most recent album 
heard per artist, whether new albums entered listening progressively.

'''

## Final Feature Engineered Table

In [ ]:
conn.execute("DROP TABLE feature_engineered_table")

In [ ]:
conn.execute("CREATE TABLE feature_engineered_table AS" \
" SELECT p.master_metadata_album_artist_name, p.play_count_per_artist, u.unique_albums_for_artist, u.num_unique_songs_for_artist, c.completion_rate, i.intentional_listening_rate, s.skip_rate,l.listening_consistency,n.num_true_albums, r.revisit_rate" \
" FROM play_count p" \
" LEFT JOIN unique_songs_and_artists_per_artist u ON p.master_metadata_album_artist_name = u.master_metadata_album_artist_name" \
" LEFT JOIN completion_rate c ON p.master_metadata_album_artist_name = c.master_metadata_album_artist_name" \
" LEFT JOIN intentional_listening_rate i ON p.master_metadata_album_artist_name = i.master_metadata_album_artist_name" \
" LEFT JOIN skip_rate s ON p.master_metadata_album_artist_name = s.master_metadata_album_artist_name" \
" LEFT JOIN listening_consistency l ON p.master_metadata_album_artist_name = l.master_metadata_album_artist_name" \
" LEFT JOIN non_single_albums_per_artist n ON p.master_metadata_album_artist_name=n.master_metadata_album_artist_name" \
" LEFT JOIN revisit_rate r ON p.master_metadata_album_artist_name=r.master_metadata_album_artist_name")

In [ ]:
conn.execute("SELECT * FROM feature_engineered_table").df().info()

,master_metadata_album_artist_name,play_count_per_artist,unique_albums_for_artist,num_unique_songs_for_artist,completion_rate,intentional_listening_rate,skip_rate,listening_consistency,num_true_albums
0,Jonas Blue,8,3,4,0.000000,0.375000,0.125000,7,1
1,Calvin Harris,198,13,17,0.555556,0.151515,0.186869,41,4
2,Sage The Gemini,10,3,4,0.400000,0.300000,0.100000,6,1
3,Tate McRae,81,13,16,0.518519,0.049383,0.259259,25,2
4,Shelly,159,2,3,0.566038,0.157233,0.358491,25,1
5,ROLE MODEL,153,11,20,0.300654,0.189542,0.359477,37,4
6,Hailee Steinfeld,252,9,9,0.583333,0.222222,0.031746,45,2
7,H.E.R.,91,5,5,0.428571,0.329670,0.472527,19,1
8,Mike Posner,109,3,5,0.431193,0.146789,0.082569,25,2
9,Internet Money,40,4,7,0.300000,0.125000,0.300000,17,1
